# Customer Churn Prediction | End-to-End Machine Learning

**Business question:** Which telecom customers are most likely to churn, and how can a retention team prioritize outreach?

**Dataset:** IBM sample Telco Customer Churn (`blastchar/telco-customer-churn` on Kaggle), 7,043 customer records.

**Workflow:** data validation → EDA → stratified train/test split → leakage-safe preprocessing → cross-validated model comparison → validation-based threshold selection → **one final test evaluation** → interpretation.

**What counts as success?** We compare **ROC-AUC** and **average precision (PR-AUC summary)** for discrimination, and **precision / recall / F1** for outreach decisions. We choose models and thresholds without looking at the held-out test set. High recall is useful, but false alarms also cost money.

> This notebook reorganizes and strengthens an exploratory notebook. Earlier test metrics were examined during exploration; for a truly unbiased final portfolio estimate, rerun this workflow from scratch on a fresh untouched holdout or obtain a new external validation dataset. The final split below is reproducible, but is not strictly unseen historically.

## 1 · Imports and reproducibility

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay, precision_recall_curve)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="deep")

## 2 · Load the dataset

The notebook checks `data/` first; if there is no local CSV, it uses `kagglehub` to download the public dataset. If the download fails, place the Kaggle CSV into `data/` manually. **No filename needs to be memorized.**

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
        csv_files = sorted(downloaded.glob("*.csv"))
    except Exception as exc:
        raise FileNotFoundError("Place the Telco Churn CSV inside data/ or install kagglehub.") from exc
if len(csv_files) != 1:
    raise ValueError(f"Expected one CSV; found {len(csv_files)}: {csv_files}")
df = pd.read_csv(csv_files[0])
print(f"Loaded {csv_files[0].name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 3 · Data quality checks and cleaning

In [ ]:
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
print("Missing before conversion:\n", df.isna().sum()[lambda s: s.gt(0)])

# Blank strings in TotalCharges become NaN; investigate before filling.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
missing_total = df.loc[df["TotalCharges"].isna(), ["tenure", "MonthlyCharges", "TotalCharges"]]
print("Missing TotalCharges:", len(missing_total))
display(missing_total)

# In this sample dataset, missing TotalCharges occur for tenure=0 (new accounts).
# Use 0 as a transparent domain assumption; verify it for another dataset.
if not (missing_total["tenure"] == 0).all():
    raise ValueError("Some missing TotalCharges have tenure > 0; investigate before imputing.")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
assert df["customerID"].is_unique
assert set(df["Churn"]) == {"Yes", "No"}
print("Missing values remaining:", df.isna().sum().sum())

## 4 · Exploratory data analysis

**Important distinction:** categorical features are compared by **within-category churn rates**, not raw bar counts, because groups have different sizes. Numeric features are compared by boxplots and distribution summaries. These are associations, not causes.

In [ ]:
churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"])
churn_rates = churn_counts / churn_counts.sum() * 100
print(pd.DataFrame({"customers": churn_counts, "percent": churn_rates.round(2)}))
fig, ax = plt.subplots(figsize=(5, 3))
sns.barplot(x=churn_counts.index, y=churn_counts.values, ax=ax)
ax.set(title="Target distribution", xlabel="Churn", ylabel="Customers")
plt.tight_layout()
plt.show()

In [ ]:
categorical_eda = ["Contract", "InternetService", "PaymentMethod", "TechSupport",
                   "Partner", "Dependents", "SeniorCitizen", "gender", "PhoneService",
                   "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                   "StreamingTV", "StreamingMovies", "PaperlessBilling"]

# Churn rates displayed for all categories with group sample sizes.
rate_tables = []
for feature in categorical_eda:
    table = pd.crosstab(df[feature], df["Churn"])
    report = pd.DataFrame({"feature": feature, "category": table.index.astype(str),
                           "n": table.sum(axis=1).values,
                           "churn_rate_pct": (100 * table.get("Yes", 0) / table.sum(axis=1)).round(2).values})
    rate_tables.append(report)
churn_by_category = pd.concat(rate_tables, ignore_index=True)
display(churn_by_category.sort_values(["feature", "churn_rate_pct"], ascending=[True, False]))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flat, ["Contract", "InternetService", "PaymentMethod", "TechSupport"]):
    subset = churn_by_category.query("feature == @feature").sort_values("churn_rate_pct")
    sns.barplot(data=subset, x="churn_rate_pct", y="category", ax=ax)
    ax.set(title=f"Churn rate by {feature}", xlabel="Churn rate (%)", ylabel="")
    ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feature in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.boxplot(data=df, x="Churn", y=feature, order=["No", "Yes"], ax=ax)
    ax.set(title=f"{feature} vs. Churn", xlabel="Churn")
plt.tight_layout()
plt.show()
display(df.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]].median())

### EDA interpretation

In the original exploratory run, higher churn was observed for month-to-month contracts (~42.7%), fiber-optic service (~41.9%), electronic check payments (~45.3%), no tech support (~41.6%), and no online security (~41.8%). Churning customers also tended to have shorter tenure and higher monthly charges. The unusually repeated 7.4% rate for **“No internet service”** across internet-related add-ons corresponds to the *same underlying group*, not six independent findings.

**Caution:** Differences are descriptive. They may reflect confounding (for example, contract type and tenure), and do not demonstrate that changing a service will cause churn to fall.

## 5 · Train / held-out test split

We keep the target as 0/1 (`Yes = 1`) and drop the ID from features. Stratification preserves class proportions. **All subsequent model selection uses only the training partition.**

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn %:", round(y_train.mean() * 100, 2),
      "Test churn %:", round(y_test.mean() * 100, 2))

## 6 · Leakage-safe preprocessing and candidates

Numeric features are imputed and standardized for linear models; categorical features are imputed and one-hot encoded. The preprocessing is **inside the pipeline**, so each cross-validation fold learns preprocessing only from its own training fold. SeniorCitizen is a coded category.

In [ ]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

def make_pipeline(estimator):
    return Pipeline([("preprocess", preprocessor), ("model", estimator)])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Balanced Logistic": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(min_samples_leaf=15, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE),
}
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.05,
                                        subsample=0.9, colsample_bytree=0.9,
                                        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=2)
except ImportError:
    print("XGBoost not installed; skipping optional candidate. pip install xgboost")

## 7 · Compare models using 5-fold cross-validation

We compare **out-of-fold averages**, not repeatedly evaluating on the held-out test set. For an imbalanced target, include both ROC-AUC and average precision. Average precision summarizes the precision-recall curve and is often reported as a PR-AUC-style metric.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "avg_precision": "average_precision",
           "f1": "f1", "precision": "precision", "recall": "recall"}
results = []
for name, estimator in models.items():
    scores = cross_validate(make_pipeline(estimator), X_train, y_train,
                            scoring=scoring, cv=cv, n_jobs=1)
    row = {"model": name}
    row.update({metric: scores[f"test_{metric}"].mean() for metric in scoring})
    row["roc_auc_std"] = scores["test_roc_auc"].std()
    results.append(row)
comparison = pd.DataFrame(results).sort_values("avg_precision", ascending=False)
display(comparison.style.format({c: "{:.3f}" for c in comparison.select_dtypes("number").columns}))

## 8 · Tune a shortlist using cross-validation

We tune **Logistic Regression and XGBoost (if available)** rather than blindly searching every model. The search optimizes **average precision**; the final operating threshold is handled separately. If XGBoost is unavailable, the notebook still works.

In [ ]:
search_spaces = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {"model__C": [0.03, 0.1, 1, 3, 10], "model__class_weight": [None, "balanced"]}
    )
}
if "XGBoost" in models:
    search_spaces["XGBoost"] = (
        models["XGBoost"],
        {"model__max_depth": [2, 3, 4], "model__min_child_weight": [1, 5],
         "model__learning_rate": [0.03, 0.1]}
    )
searches = {}
for name, (model, grid) in search_spaces.items():
    search = GridSearchCV(make_pipeline(model), grid, scoring="average_precision",
                          cv=cv, n_jobs=1, refit=True)
    search.fit(X_train, y_train)
    searches[name] = search
    print(f"{name}: CV average precision={search.best_score_:.3f}; {search.best_params_}")

best_name = max(searches, key=lambda name: searches[name].best_score_)
best_pipeline = searches[best_name].best_estimator_
print("Selected by training CV:", best_name)

## 9 · Select a threshold using **out-of-fold training predictions**

Thresholds trade off **precision** (avoid costly false alarms) and **recall** (catch churners). We choose a threshold by maximizing out-of-fold F1, but a real retention team could specify a contact budget, minimum precision, or per-contact cost instead. **Do not choose a threshold using the test set.**

In [ ]:
oof_probability = cross_val_predict(best_pipeline, X_train, y_train,
                                   cv=cv, method="predict_proba", n_jobs=1)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, oof_probability)
f1_by_threshold = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_ix = int(np.argmax(f1_by_threshold))
best_threshold = float(thresholds[best_ix])
print(f"Chosen threshold (training OOF): {best_threshold:.3f}")
print(f"OOF precision={precision[best_ix]:.3f}, recall={recall[best_ix]:.3f}, F1={f1_by_threshold[best_ix]:.3f}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, precision[:-1], label="Precision")
ax.plot(thresholds, recall[:-1], label="Recall")
ax.plot(thresholds, f1_by_threshold, label="F1")
ax.axvline(best_threshold, color="black", linestyle="--", label="Chosen threshold")
ax.set(xlabel="Prediction threshold", ylabel="Score", title="Training-only threshold selection")
ax.legend()
plt.tight_layout()
plt.show()

## 10 · Final held-out evaluation — run once after selection

The following is the **final** evaluation on the reserved test set. Do not keep changing features, hyperparameters, or thresholds to improve these test results. If you iterate further, use training cross-validation and reserve a new external dataset for confirmation.

In [ ]:
# Best estimator was already refit on all training data by GridSearchCV.
test_prob = best_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

final_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_prob),
    "average_precision": average_precision_score(y_test, test_prob),
}
print("Selected model:", best_name, "| threshold:", round(best_threshold, 3))
display(pd.Series(final_metrics, name="Held-out test score").to_frame().T.round(3))
print(classification_report(y_test, test_pred, target_names=["Stayed", "Churned"]))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=["Stayed", "Churned"], ax=axes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, test_prob, ax=axes[1])
PrecisionRecallDisplay.from_predictions(y_test, test_prob, ax=axes[2])
axes[0].set_title("Confusion matrix")
axes[1].set_title("ROC curve")
axes[2].set_title("Precision–Recall curve")
plt.tight_layout()
plt.show()

## 11 · Explain predictions

Use permutation importance on the held-out set as a **descriptive final report only**, not for feature selection or further model tuning. It estimates how much a model's ROC-AUC drops if a feature is shuffled; correlated features may divide importance between themselves.

In [ ]:
from sklearn.inspection import permutation_importance
importance = permutation_importance(best_pipeline, X_test, y_test,
                                    scoring="roc_auc", n_repeats=5,
                                    random_state=RANDOM_STATE, n_jobs=1)
feature_importance = pd.Series(importance.importances_mean, index=X_test.columns).sort_values(ascending=False)
display(feature_importance.head(12).rename("Mean ROC-AUC drop").to_frame())
fig, ax = plt.subplots(figsize=(8, 5))
feature_importance.head(12).sort_values().plot.barh(ax=ax)
ax.set(xlabel="Mean ROC-AUC drop on permutation", title="Model-level feature importance")
plt.tight_layout()
plt.show()

## 12 · What would a business do with this?

- **Prioritize risk-based outreach:** Contact high-risk customers, while accounting for outreach capacity and costs. The threshold is a decision rule, not a universal truth.
- **Investigate observed patterns:** Month-to-month customers and customers without certain support services exhibited higher churn rates. Validate business explanations before changing policy.
- **Measure intervention impact:** A predictive score does **not** prove that discounts, contract offers or support upgrades prevent churn. Run a randomized retention campaign (A/B test) to estimate actual lift.
- **Limitations:** This is a static sample dataset; time-aware validation, stability monitoring, fairness/privacy checks, and prospective validation would be needed for production.

### Portfolio checklist

- [x] Reproducible data checks and EDA
- [x] Leakage-safe pipelines and stratified CV
- [x] Multi-model comparison and tuned shortlist
- [x] Out-of-fold threshold selection
- [x] ROC-AUC, average precision, confusion matrix, PR curve
- [x] Feature interpretation and business recommendations
- [ ] Validate on genuinely new data / prospective cohort
- [ ] Optional Streamlit demo for risk scoring (after validating inputs and documentation)